<a href="https://colab.research.google.com/github/zamanuddinkhan/Python-AI-LLM/blob/main/LangChain_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**LangChain Basics**

**Chains**

In LangChain, a Chain is a sequence of connected steps where the output of one step becomes the input of the next step.

It helps automate multi-step AI workflows.

In [ ]:
#Install Langchain
!pip install langchain==0.1.20 langchain-community==0.0.38 langchain-openai==0.1.7 langchain-core==0.1.52 chromadb==0.4.24

In [ ]:
!pip install grandalf

In [ ]:
#setup API key
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')
OpenAI_API_key=userdata.get('OPENAI_API_KEY')
import warnings
warnings.filterwarnings('ignore')

1. Simple Chains

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import IPython.display

prompt = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)

model = ChatOpenAI(model='gpt-4o-mini',api_key=OpenAI_API_key,temperature=0)

parser = StrOutputParser()

chain = prompt | model | parser # Langchain Expression Language - | -> Pipe operator to create pipelines

result = chain.invoke({'topic': 'Paris'})
print(result)

#markdown
IPython.display.Markdown(result)

chain.get_graph().print_ascii()


**Without LCEL**

In [ ]:
from langchain.chains import LLMChain

# Prompt Template
prompt = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)

# Load Model
model = ChatOpenAI(
    model='gpt-4o-mini',
    api_key=OpenAI_API_key,
    temperature=0
)

# Output Parser
parser = StrOutputParser()

# Create Chain (Without LCEL)
chain = LLMChain(
    llm=model,
    prompt=prompt
)

# Execute Chain
result = chain.run({'topic': 'Paris'})

# Parse Output
parsed_result = parser.parse(result)

# Print Result
print(parsed_result)

# Display as Markdown
IPython.display.Markdown(parsed_result)

2. Sequential Chain

In [ ]:
prompt1 = PromptTemplate(
    template='Generate a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a 5 pointer summary for the following text:\n{text}',
    input_variables=['text']
)

model = ChatOpenAI(model='gpt-4o-mini',api_key=OpenAI_API_key,temperature=0)

parser = StrOutputParser()

chain = prompt1 | model | parser | prompt2 | model | parser

result = chain.invoke({'topic': 'Economic Recession'})

print(result)

chain.get_graph().print_ascii()

3. Parallel Chains

In [ ]:
from langchain.schema.runnable import RunnableParallel

model1 = ChatOpenAI(model='gpt-4o-mini',api_key=OpenAI_API_key,temperature=0)
model2 = ChatOpenAI(model='gpt-4o',api_key=OpenAI_API_key,temperature=0)

prompt1 = PromptTemplate(
    template='Generate short and concise notes for the following text:\n{text}',
    input_variables=['text']
)
prompt2 = PromptTemplate(
    template='Generate 5 short question and answers based on the following text:\n{text}',
    input_variables=['text']
)

parser = StrOutputParser()

prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document:\n{notes}\n{quiz}',
    input_variables=['notes', 'quiz']
)

chain1 = prompt1 | model1 | parser
chain2 = prompt2 | model2 | parser

parallel_chain = RunnableParallel({
    'notes': chain1,
    'quiz': chain2
})

merge_chain = prompt3 | model1 | parser

chain = parallel_chain | merge_chain

text = """
    Support Vector Machines (SVM) are powerful supervised learning algorithms used for classification and regression tasks.
    They work by finding an optimal hyperplane that best separates data points in a high-dimensional space.
    The goal of SVM is to maximize the margin between different classes while minimizing misclassification errors.
    For linearly separable data, SVM identifies the hyperplane that maximizes the distance between the closest points (support vectors) of each class.
    However, real-world datasets are often not linearly separable, which is where SVM uses kernel functions to project data into a higher-dimensional space where it becomes separable.
    SVM supports several kernel functions, such as linear, polynomial, radial basis function (RBF), and sigmoid kernels, allowing it to adapt to different types of data distributions.
    The RBF kernel is particularly effective in handling complex relationships by mapping data into an infinite-dimensional space.
    The algorithm relies on regularization (C parameter) to control the trade-off between achieving a low error and maintaining a large margin.
    A higher C focuses more on correct classification, potentially leading to overfitting, whereas a lower C prioritizes a wider margin and better generalization.
    For regression tasks, Support Vector Regression (SVR) follows a similar principle but aims to fit data within an ε-tube, where predictions are penalized only if they fall outside this margin.
    Despite its strength in handling high-dimensional data and avoiding overfitting, SVM can be computationally expensive, especially with large datasets, as training complexity grows significantly with sample size.
    Nonetheless, it remains a widely used algorithm in fields like image recognition, bioinformatics, and finance due to its robustness and versatility.
"""
result = chain.invoke({'text': text})
print(result)

chain.get_graph().print_ascii()

4. Conditional Chains

In [ ]:
from langchain.schema.runnable import RunnableParallel, RunnableBranch, RunnableLambda
from typing import Literal
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field

model = ChatOpenAI(model='gpt-4o-mini',api_key=OpenAI_API_key,temperature=0)
parser = StrOutputParser()

class Feedback(BaseModel):
    sentiment: Literal['positive','negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into positive or negative:\n {feedback} \n {format_instructions}',
    input_variables=['feedback'],
    partial_variables={'format_instructions': parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback:\n{feedback}',
    input_variables=['feedback']
)

chain2 = prompt2 | model | parser

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback:\n{feedback}',
    input_variables=['feedback']
)

chain3 = prompt3 | model | parser

branch_chain = RunnableBranch( # Essentially like if-else statements
    (lambda x: x.sentiment == 'positive', chain2),
    (lambda x: x.sentiment == 'negative', chain3),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

print(chain.invoke({'feedback': 'The color of shoes is not that I ordered'}))

chain.get_graph().print_ascii()


**LangChain Chat Models**

In [ ]:
# OpenAI Chat Model

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-4o',api_key=OpenAI_API_key, temperature=1.5, max_completion_tokens=100)

result = model.invoke("Write a five line poem on football.")
print(result.content)

OpenSource LLM APIs: Groq Cloud

In [ ]:
from langchain_groq import ChatGroq

from google.colab import userdata
groq_key=userdata.get('GROQ_API_KEY')

chat = ChatGroq(model="llama-3.1-8b-instant",api_key=groq_key)

response = chat.invoke("What is the capital of India ?")
print(response.content)

OpenSource LLMs: HuggingFace Platform

In [ ]:
!pip install langchain-huggingface

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs={
        'temperature': 0.6,
        'max_new_tokens': 100
    }
)

model = ChatHuggingFace(llm=llm)

result = model.invoke("What is the capital of New Zealand?")
print(result.content)

Gemini Chat Model

In [ ]:
!pip install langchain-google-genai

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
gemini_api=userdata.get('GEMINI_API_KEY')

model = ChatGoogleGenerativeAI(api_key=gemini_api,model='models/gemini-flash-latest')
result = model.invoke('What is the capital of Switzerland?')
print(result.content)

Anthropic Chat Model

In [ ]:
!pip install langchain-anthropic

In [ ]:
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv

load_dotenv()

model = ChatAnthropic(model='claude-3-5-sonnet-20241022')
result = model.invoke('What is the capital of Norway?')
print(result.content)

**LangChain Runnable**

In LangChain
, Runnables are the core abstraction used to build AI pipelines in a modular and composable way.
They represent any component that can take an input, process it, and return an output.

Runnables are heavily used in the newer LangChain architecture, especially with LCEL (LangChain Expression Language).

Major Types of Runnables

    - RunnableLambda: Wraps any Python function or lambda as a runnable object.

    - RunnableParallel: Executes a dictionary of runnables in parallel, applying the same input to each and returning a map of outputs.

    - RunnablePassthrough: Passes data through unchanged or with additional keys, which helps when chaining transformations.

    - RunnableSequence: Chains a sequence of runnables so the output from one is the input for the next, useful for building multi-step workflows.

    - RunnableBranch: Selects a particular branch (runnable) based on a runtime condition, enabling conditional logic in pipelines.


In [ ]:
!pip install langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os, warnings
warnings.filterwarnings('ignore')

from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')

**1. RunnableSequence**

In [ ]:
from langchain.schema.runnable import RunnableSequence

# Initialize the chat model
chat_model = ChatOpenAI(api_key=openai_api_key, temperature=0.7)

# Define the prompts
prompt1 = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Explain the following joke: {text}',
    input_variables=['text']
)

# Define the output parser
output_parser = StrOutputParser()

chain = RunnableSequence(prompt1,chat_model,output_parser,prompt2,chat_model,output_parser)
print(chain.invoke({'topic': 'AI'}))

**2. Runnable Parallel**

In [ ]:
from langchain.schema.runnable import RunnableSequence, RunnableParallel
import IPython

prompt1 = PromptTemplate(
    template='Generate a tweet about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a LinkedIn post about {topic}',
    input_variables=['topic']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'tweet': RunnableSequence(prompt1, chat_model, parser),
    'linkedin_post': RunnableSequence(prompt2, chat_model, parser)
})

result = parallel_chain.invoke({'topic': 'AI'})
print("Tweet:", result['tweet'])
print("\nLinkedIn Post:", result['linkedin_post'])


3. Runnable Branch

In [ ]:
#import runnable
from langchain.schema.runnable import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch

from google.colab import userdata
os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

# Initialize the chat model
chat_model = ChatOpenAI(api_key=os.environ['OPENAI_API_KEY'])

# Define the prompt
#Branch1
prompt1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

#Branch2
prompt2 = PromptTemplate(
    template='Summarize the following text: \n{text}',
    input_variables=['text']
)

# Define the output parser
output_parser = StrOutputParser()

report_gen_chain = prompt1 | chat_model | output_parser

print(report_gen_chain.invoke({'topic': 'Russia vs. Ukraine'}))


branch_chain = RunnableBranch(
    (lambda x: len(x.split()) > 300, prompt2 | chat_model |output_parser),
    RunnablePassthrough()
)

final_chain = report_gen_chain | branch_chain

print("Summary report: ",final_chain.invoke({'topic': 'Russia vs. Ukraine'}))

**4. Runnable Lambda**

In [ ]:
# Define the prompt
prompt = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

def word_count(text):
    return len(text.split())

# Define the output parser
output_parser = StrOutputParser()

joke_gen_chain = RunnableSequence(prompt,chat_model,output_parser)

parallel_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'word_count': RunnableLambda(lambda x: len(x.split()))
    #Alternate
    #runnable_word_counter = RunnableLambda(word_counter)
})

final_chain = RunnableSequence(joke_gen_chain,parallel_chain)

result = final_chain.invoke({'topic': 'AI'})

final_result = """{} \n Word Count: {}""".format(result['joke'], result['word_count'])
print(final_result)


# from langchain.schema.runnable import RunnableLambda

# def word_counter(text):
#     return len(text.split())

# runnable_word_counter = RunnableLambda(word_counter)
# print(runnable_word_counter.invoke('Hi there, how are you?'))

5. Runnable Passthrough

In [ ]:
# Define the prompts
prompt1 = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Explain the following joke: {text}',
    input_variables=['text']
)

parser = StrOutputParser()

joke_gen_chain = RunnableSequence(prompt1,chat_model,parser)

parallel_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'explanation': RunnableSequence(prompt2,chat_model,parser)
})

final_chain = RunnableSequence(joke_gen_chain,parallel_chain)

print(final_chain.invoke({'topic': 'football'}))

**LangChain PromptTemplates**

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

A Prompt Template in LangChain is a blueprint for dynamically creating prompts. Instead of hardcoding a string prompt, you define a template with variables, and LangChain fills those variables at runtime.

Think of it as:

“A reusable prompt structure that generates text dynamically based on inputs.”

**Basic Prompts Template**

In [ ]:
!pip install langchain_groq

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["language", "text"],
    template="Translate the following text to {language}:\n{text}"
)

prompt_text = prompt.format(language="Spanish", text="I love programming.")
print(prompt_text)

In [ ]:
from langchain_groq import ChatGroq

from google.colab import userdata
groq_key=userdata.get('GROQ_API_KEY')

chat = ChatGroq(model="llama-3.1-8b-instant",api_key=groq_key)

response = chat.invoke(prompt_text)
print(response.content)

ChatPromptTemplate (for ChatModels)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    ("human", "Explain {topic} in simple terms.")
])

chat = ChatGroq(model="llama-3.1-8b-instant",api_key=groq_key)
response = chat.invoke(prompt.format_messages(topic="Machine Learning"))
print(response.content)

FewShot Prompt Templates

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}"
)

examples = [
    {"input": "2 + 2", "output": "4"},
    {"input": "3 + 7", "output": "10"}
]

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Input: {input}\nOutput:",
    input_variables=["input"]
)

print(few_shot_prompt.format(input="5 + 8"))

In [ ]:
chat = ChatGroq(model="llama-3.1-8b-instant",api_key=groq_key)
response = chat.invoke(few_shot_prompt.format(input="5 + 8"))
print(response.content)

**LangChain- Memory**

In LangChain, Memory is how a system remembers past interactions between the user and the model.

Without memory, every call to the LLM is stateless — it forgets what happened before. With memory, your agent can hold context, recall facts, and behave coherently across turns.

**Types of Memory**

- ConversationBufferMemory

- ConversationBufferWindowMemory

- ConversationTokenBufferMemory

- ConversationSummaryMemory

**ConversationBufferMemory**

Concept: Stores all messages from the conversation in order (like a running transcript). When a new question comes in, it passes the entire chat history back to the LLM.

Use when: You want the model to always have full context of the conversation.

In [ ]:
!pip install langchain==0.1.20 langchain-community==0.0.38 langchain-openai==0.1.7 langchain-core==0.1.52

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

In [ ]:
import os

from google.colab import userdata
os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

In [ ]:
llm = ChatOpenAI(temperature=0.0)  #defualt model gpt-4o-mini
memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=llm,
    verbose=True,
    memory = memory  #Attached bufer memory, allow us to store the whole conversation
)

In [ ]:
llm.invoke(input="Hi my name is Ajith, I am an AI architect, I live in India")

In [ ]:
llm.invoke(input="Explain NLP in simple terms with applications")

In [ ]:
llm.invoke(input="What is my name ?")